# 量价延续/反转信号 — 最小示例

研究用模块，**不接实盘**。输入 CSV 或 DataFrame（`ts,symbol,tf,open,high,low,close,volume,oi`）。

流程：`load_klines` → `generate_signals` → `run_volume_price_backtest`

In [ ]:
from pathlib import Path
import pandas as pd
from oi_mornitor.volume_price import load_klines, generate_signals, run_volume_price_backtest

csv_path = Path('sample_klines.csv')
raw = load_klines(csv_path)
raw.head()

In [ ]:
h1 = (
    raw.assign(bucket=raw['ts'] // 3_600_000)
    .groupby('bucket', as_index=False)
    .agg(
        ts=('bucket', 'first'),
        symbol=('symbol', 'first'),
        tf=('tf', lambda _: '1h'),
        open=('open', 'first'),
        high=('high', 'max'),
        low=('low', 'min'),
        close=('close', 'last'),
        volume=('volume', 'sum'),
    )
)
h1['ts'] = h1['bucket'] * 3_600_000
h1 = h1.drop(columns=['bucket'])

full = pd.concat([raw, h1], ignore_index=True)
signals, enriched = generate_signals(full, signal_tfs=('15m',))
print('signals', len(signals))
enriched[['ts','bar_class','vol_z','range_z','efficiency','h1_position']].tail(8)

In [ ]:
report = run_volume_price_backtest(enriched, signals)
report.to_dict()